Setup:
Imports and the seasons to pull data from.

In [21]:
import nflreadpy as nfl
import pandas as pd

SEASONS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Game schedules:
One row per game. 
"results" is a home score minus away score.
"spread line" is the closing line from the home side.
Home team covers when result > spread_line.

In [22]:
games = nfl.load_schedules(SEASONS).to_pandas()
print(games.shape)
games.head()

(2227, 46)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2018_01_ATL_PHI,2018,REG,1,2018-09-06,Thursday,20:20,ATL,12,PHI,...,8.0,00-0026143,00-0029567,Matt Ryan,Nick Foles,Dan Quinn,Doug Pederson,John Hussey,PHI00,Lincoln Financial Field
1,2018_01_BUF_BAL,2018,REG,1,2018-09-09,Sunday,13:00,BUF,3,BAL,...,12.0,00-0033958,00-0026158,Nathan Peterman,Joe Flacco,Sean McDermott,John Harbaugh,Shawn Hochuli,BAL00,M&T Bank Stadium
2,2018_01_PIT_CLE,2018,REG,1,2018-09-09,Sunday,13:00,PIT,21,CLE,...,11.0,00-0022924,00-0028118,Ben Roethlisberger,Tyrod Taylor,Mike Tomlin,Hue Jackson,Shawn Smith,CLE00,FirstEnergy Stadium
3,2018_01_CIN_IND,2018,REG,1,2018-09-09,Sunday,13:00,CIN,34,IND,...,NaN,00-0027973,00-0029668,Andy Dalton,Andrew Luck,Marvin Lewis,Frank Reich,Peter Morelli,IND00,Lucas Oil Stadium
4,2018_01_TEN_MIA,2018,REG,1,2018-09-09,Sunday,13:00,TEN,20,MIA,...,7.0,00-0032268,00-0029701,Marcus Mariota,Ryan Tannehill,Mike Vrabel,Adam Gase,Jerome Boger,MIA00,Hard Rock Stadium


Columns that matter:
Rest days come free here, so that feature is already done.

In [23]:
games[["game_id", "season", "week", "home_team", "away_team",
       "result", "spread_line", "home_rest", "away_rest"]].head()

,game_id,season,week,home_team,away_team,result,spread_line,home_rest,away_rest
0,2018_01_ATL_PHI,2018,1,PHI,ATL,6,1.0,7,7
1,2018_01_BUF_BAL,2018,1,BAL,BUF,44,7.5,7,7
2,2018_01_PIT_CLE,2018,1,CLE,PIT,0,-3.5,7,7
3,2018_01_CIN_IND,2018,1,IND,CIN,-11,-1.0,7,7
4,2018_01_TEN_MIA,2018,1,MIA,TEN,7,-1.0,7,7


Play by play:
EPA source is here. Selecting columns inside the polars call because the full table is 372 columns wide. 

In [24]:
pbp = nfl.load_pbp(SEASONS).select([
    "game_id", "season", "week", "season_type",
    "posteam", "defteam", "epa", "pass", "rush"
]).to_pandas()
print(pbp.shape)

(389358, 9)


Cache to disk:
Saved as a parquet in data/ so kernel restarts do not redownload Gitignored.

In [25]:
import os
os.makedirs("data", exist_ok=True)
pbp.to_parquet("data/pbp.parquet")
games.to_parquet("data/games.parquet")

Types of plays:
Removes kickoffs, punts, penalties, and kneels. 

In [26]:
plays = pbp[(pbp["season_type"] == "REG") &
            ((pbp["pass"] == 1) | (pbp["rush"] == 1))]
plays = plays.dropna(subset=["posteam", "defteam", "epa"])
print(plays.shape)

(276770, 9)


Team game aggregates:
Filter to pass and run pkays, then average EPA per team per game from both the offensive and defensive side. 
"plays" is the pace feature.

In [27]:
off = (plays.groupby(["game_id", "season", "week", "posteam"])
             .agg(off_epa=("epa", "mean"), plays=("epa", "size"))
             .reset_index()
             .rename(columns={"posteam": "team"}))

deff = (plays.groupby(["game_id", "season", "week", "defteam"])
              .agg(def_epa=("epa", "mean"))
              .reset_index()
              .rename(columns={"defteam": "team"}))

team_games = off.merge(deff, on=["game_id", "season", "week", "team"])
print(team_games.shape)
team_games.head()

(4254, 7)


,game_id,season,week,team,off_epa,plays,def_epa
0,2018_01_ATL_PHI,2018,1,ATL,-0.209804,70,-0.073813
1,2018_01_ATL_PHI,2018,1,PHI,-0.073813,71,-0.209804
2,2018_01_BUF_BAL,2018,1,BAL,0.051798,80,-0.614689
3,2018_01_BUF_BAL,2018,1,BUF,-0.614689,65,0.051798
4,2018_01_CHI_GB,2018,1,CHI,-0.002186,70,-0.035099


Rolling form features:
8 game rolling average of each team's EPA and pace, shifted by one game so the current game never leaks into its own prediction.

In [28]:
team_games = team_games.sort_values(["team", "season", "week"])
WINDOW = 8

for col in ["off_epa", "def_epa", "plays"]:
    team_games[f"r_{col}"] = (
        team_games.groupby("team")[col]
        .transform(lambda s: s.shift(1).rolling(WINDOW, min_periods=4).mean())
    )

team_games.head(12)

,game_id,season,week,team,off_epa,plays,def_epa,r_off_epa,r_def_epa,r_plays
30,2018_01_WAS_ARI,2018,1,ARI,-0.212811,52,0.180503,NaN,NaN,NaN
32,2018_02_ARI_LA,2018,2,ARI,-0.416535,45,0.180278,NaN,NaN,NaN
66,2018_03_CHI_ARI,2018,3,ARI,-0.217478,50,-0.053186,NaN,NaN,NaN
120,2018_04_SEA_ARI,2018,4,ARI,-0.083162,60,0.033652,NaN,NaN,NaN
126,2018_05_ARI_SF,2018,5,ARI,-0.101366,47,-0.155786,-0.232497,0.085312,51.750
156,2018_06_ARI_MIN,2018,6,ARI,-0.351903,58,-0.155635,-0.206270,0.037092,50.800
196,2018_07_DEN_ARI,2018,7,ARI,-0.644318,66,0.045789,-0.230543,0.004971,52.000
236,2018_08_SF_ARI,2018,8,ARI,-0.057073,66,-0.129249,-0.289653,0.010802,54.000
268,2018_10_ARI_KC,2018,10,ARI,-0.127965,72,0.084661,-0.260581,-0.006704,55.500
312,2018_11_OAK_ARI,2018,11,ARI,-0.226196,54,-0.013130,-0.249975,-0.018685,58.000


Join form to schedule:
Attach each team's pre-game form to the game they are about to play, once as home and once as away.

In [29]:
form = team_games[["game_id", "team", "r_off_epa", "r_def_epa", "r_plays"]]

df = games[games["game_type"] == "REG"].dropna(subset=["result", "spread_line"])

df = df.merge(form.add_prefix("home_"),
               left_on=["game_id", "home_team"],
               right_on=["home_game_id", "home_team"], how="left")

df = df.merge(form.add_prefix("away_"),
              left_on=["game_id", "away_team"],
              right_on=["away_game_id", "away_team"], how="left")

print(df.shape)

(2127, 54)


Difference features and targets:
Home minus away for each stat, since what decides a game is the gap between the two teams rather than either team's raw number.

In [37]:
df["d_off_epa"] = df["home_r_off_epa"] - df["away_r_off_epa"]
df["d_def_epa"] = df["home_r_def_epa"] - df["away_r_def_epa"]
df["d_pace"] = df["home_r_plays"] - df["away_r_plays"]
df["d_rest"] = df["home_rest"] - df["away_rest"]

df["home_won"] = (df["result"] > 0).astype(int)
df["home_covered"] = (df["result"] > df["spread_line"]).astype(int)

FEATURES = ["d_off_epa", "d_def_epa", "d_pace", "d_rest"]
model_df = df.dropna(subset=FEATURES)
print(model_df.shape)

(2034, 60)


In [38]:
print("rows:        ", len(model_df))
print("cover rate:  ", round(model_df["home_covered"].mean(), 3))
print("home win rate:", round(model_df["home_won"].mean(), 3))

rows:         2034
cover rate:   0.472
home win rate: 0.534
